# Graph Neural Networks for Predicting Chromatin Activation During Craniofacial Development

**Author:** Yousra Ben Zouari  
**Context:** Follow-up computational analysis to [Ben Zouari et al., *Nature Communications* 2024](https://doi.org/10.1038/s41467-024-53613-7)  
**Paper:** *Polycomb Chromatin Topology Enables Long-Range Enhancer Recruitment during Craniofacial Development*

---

## 1. Introduction & Biological Context

During craniofacial development, cranial neural crest cells (CNCCs) delaminate from the neural tube and migrate to populate the facial prominences, where they undergo massive transcriptional reprogramming. Our *Nature Communications* paper characterized how Polycomb-mediated chromatin topology orchestrates long-range enhancer recruitment during this developmental transition.

A key observation from the paper was that many regulatory elements — both gene promoters and distal-acting enhancers (DAEs) — exist in non-active epigenetic states at E8.5 (pre-migration) and transition to an active state by E10.5 (post-migration). These non-active states include:

- **Bivalent**: marked by both H3K27me3 (Polycomb) and H3K4me2 (Trithorax) — poised for rapid activation
- **Poised**: H3K27me3+ only at moderate levels
- **Pc-only**: strong H3K27me3, no active marks
- **Primed**: low-level H3K4me2 without activation marks
- **Negative**: no detectable histone modifications

### The Question

**Can we predict which non-active regulatory elements at E8.5 will become active by E10.5, using their epigenetic features and their position in the chromatin interaction network?**

We model this as a **node classification** problem on a graph, using Graph Neural Networks (GNNs) — a natural fit since our data contains both node features (ChIP-seq signals) and graph structure (CHiCAGO-scored chromatin interactions from Promoter Capture Hi-C).

## 2. Data Description

Data was extracted from SummarizedExperiment R objects containing Promoter Capture Hi-C and ChIP-seq data at two developmental timepoints (E8.5 and E10.5).

### 2.1 Graph Structure

| Property | Value |
|----------|-------|
| **Total nodes** | 108,162 |
| Promoters | 39,707 (37%) |
| Enhancers (DAEs) | 68,455 (63%) |
| **Edges** (CHiCAGO score ≥ 5) | ~39,836 significant interactions at E8.5 |
| **All edges** (score > 0) | ~110,986 interaction pairs |
| Mean degree (threshold ≥ 5) | 1.7 — most nodes are isolated |
| Isolated nodes (no edges) | ~84.8% at threshold 5 |

### 2.2 Node Features Available

Three histone modification ChIP-seq signals measured at E8.5:
- **H3K27ac**: marks active enhancers and promoters
- **H3K4me2**: Trithorax group mark, present at poised and active elements  
- **H3K27me3**: Polycomb repressive mark

Additionally available but **excluded from features** after careful consideration:
- **RNA-seq expression**: only meaningful for promoters (enhancers have minimal eRNA signal that doesn't reflect regulatory state), creating a confounding feature that encodes node type rather than activation potential

### 2.3 Label Scheme

| Label | Meaning | Count | Percentage |
|-------|---------|-------|------------|
| 1 (positive) | Non-active at E8.5 → Active at E10.5 | ~14,312 | 19% of labeled |
| 0 (negative) | Non-active at E8.5 → Remains non-active | ~60,994 | 81% of labeled |
| -1 (masked) | Already active at E8.5 or unknown | ~32,856 | Excluded |

This ~1:4.3 class imbalance is an important design constraint — it affects loss function choice, evaluation metrics, and decision threshold.

## 3. Technical Approach

### 3.1 Why Graph Neural Networks?

GNNs are a natural choice because our data has both:
- **Node features** (ChIP-seq signals at each regulatory element)
- **Graph topology** (CHiCAGO-scored chromatin interactions between elements)

GNNs learn representations through **message passing**: each node aggregates information from its neighbors, allowing the model to capture how a node's local chromatin environment might influence its activation fate. After *k* layers of message passing, each node's representation incorporates information from its *k*-hop neighborhood.

### 3.2 Handling Class Imbalance

With 19% positives, we employed:
- **Class-weighted cross-entropy loss**: weights inversely proportional to class frequency, so the minority class contributes equally to the gradient
- **AUPRC as primary metric**: Area Under the Precision-Recall Curve is more informative than AUROC for imbalanced problems, as it directly measures performance on the minority class
- **Early stopping on validation AUPRC** (patience=30): prevents overfitting and selects the model that best discriminates the minority class

### 3.3 Train/Val/Test Split

70% / 15% / 15% **stratified** split to preserve class ratios across sets. Only labeled nodes (y ≥ 0) are used for training and evaluation, but **all nodes participate in message passing** (transductive learning).

### 3.4 Learning Rate Scheduling

`ReduceLROnPlateau` monitoring AUPRC with patience=10 and factor=0.5. This allows the model to fine-tune when progress stalls.

## 4. Experimental Progression

We conducted a systematic series of experiments, each testing a specific hypothesis about what drives prediction performance. The progression follows the scientific method: observe a result, form a hypothesis, test it, and iterate.

---

### Experiment 1: Baseline GAT with Binary Edges

**Hypothesis:** A Graph Attention Network can learn to predict activation from ChIP-seq features combined with chromatin interaction topology.

**Design choices:**
- **Architecture:** GAT (Graph Attention Network) — chosen because the attention mechanism can learn which chromatin neighbors are most informative, rather than weighting all edges equally (as GCN does)
- **Layers:** 3 (allows information to propagate up to 3 hops in the interaction graph)
- **Attention heads:** 4 (multi-head attention captures different types of neighbor importance)
- **Hidden channels:** 64 per head
- **Node features:** 3 ChIP-seq signals (H3K27ac, H3K4me2, H3K27me3), z-score standardized
- **Edges:** Binary — CHiCAGO score ≥ 5 creates an edge, undirected
- **Self-loops:** Added to all nodes so isolated nodes still process their own features
- **Dropout:** 0.3 to prevent overfitting

**Why exclude expression?** RNA-seq expression is only biologically meaningful for promoters. For enhancers (~63% of nodes), expression is near-zero, effectively encoding node type rather than regulatory potential. Including it would create a confounding feature.

In [ ]:
from IPython.display import Image, display
import os

fig_base = '../figures'

print("=" * 70)
print("EXPERIMENT 1: Baseline GAT — Training Curves")
print("=" * 70)
display(Image(filename=f'{fig_base}/exp1_baseline/training_curves.png', width=900))

In [ ]:
print("ROC and Precision-Recall Curves")
display(Image(filename=f'{fig_base}/exp1_baseline/roc_pr_curves.png', width=800))

In [ ]:
print("Confusion Matrix")
display(Image(filename=f'{fig_base}/exp1_baseline/confusion_matrix.png', width=500))

In [ ]:
print("UMAP of Learned Node Embeddings")
print("Left: Activation label | Center: E8.5 epigenetic state | Right: Promoter vs Enhancer")
display(Image(filename=f'{fig_base}/exp1_baseline/embeddings_umap.png', width=900))

In [ ]:
print("UMAP Colored by Histone Mark Levels")
display(Image(filename=f'{fig_base}/exp1_baseline/embeddings_umap_chipseq.png', width=900))

In [ ]:
print("Graph Degree Distribution")
display(Image(filename=f'{fig_base}/exp1_baseline/graph_statistics.png', width=900))

**Results:**

| Metric | Value |
|--------|-------|
| AUROC | 0.751 |
| AUPRC | 0.382 (random baseline = 0.190) |
| F1 | 0.423 |
| Precision | 0.279 |
| Recall | 0.876 |
| Best epoch | 127 |
| Parameters | 87,714 |

**Interpretation:**
- The model learns real signal: AUPRC is 2× the random baseline
- However, it exhibits **high recall / low precision** — it catches 87.6% of true activations but with 4,864 false positives
- The class-weighted loss pushes the decision boundary aggressively toward predicting "activated"
- The UMAP shows the model creates a meaningful embedding space: the Pc-only cluster is well-separated, and H3K27ac shows a clear spatial gradient

**Key observation from the graph statistics:** Mean degree = 1.7, and 84.8% of nodes are isolated. For these isolated nodes, the GNN degrades to a simple MLP — there is no graph structure to exploit. This led us to hypothesize that **graph topology might be a bottleneck**.

---

### Experiment 2: Testing the Topology Hypothesis

**Hypothesis:** The sparse graph (84.8% isolated nodes) is limiting the GNN. We can improve performance by either (a) adding more edges or (b) focusing only on nodes where topology exists.

**Two sub-experiments:**

#### 2a. Lower CHiCAGO Threshold (3 instead of 5)

**Rationale:** Lowering the significance threshold from 5 → 3 triples the number of edges (~80K → ~242K), giving the GNN much denser neighborhoods for message passing. The trade-off is noisier edges.

#### 2b. Connected Subgraph Only

**Rationale:** If topology helps, the model should perform better when trained only on nodes that actually have edges (16K connected nodes instead of 108K total). This is the strongest test of whether message passing adds value.

In [ ]:
print("=" * 70)
print("EXPERIMENT 2: Topology Tuning — Comparison")
print("=" * 70)
display(Image(filename=f'{fig_base}/exp2_topology/comparison_chart.png', width=700))

**Results:**

| Variant | Nodes | Edges | AUROC | AUPRC | F1 |
|---------|-------|-------|-------|-------|----|
| Baseline (threshold=5) | 108,162 | ~80K | 0.751 | 0.382 | 0.423 |
| Lower threshold (=3) | 108,162 | ~242K | 0.751 | 0.383 | 0.421 |
| Connected subgraph only | 16,416 | ~96K | 0.744 | 0.096 | 0.096 |

**Interpretation:**

- **Threshold = 3**: Virtually identical to baseline. Adding ~160K noisy edges had zero impact. The extra interactions at CHiCAGO 3-5 don't carry activation-predictive topology signal.

- **Connected only**: AUPRC collapsed to 0.096 (below random baseline of 0.19 for the full graph). This happened because the connected subgraph has an even more extreme class imbalance — elements involved in CHiCAGO interactions are predominantly stable regulatory elements in constitutive loops, not elements that switch state during development.

**Conclusion:** Graph topology manipulation does not improve prediction. The signal is not in the edges. This is the first indication that **chromatin 3D topology at E8.5 may not be predictive of activation fate**.

---

### Experiment 3: Continuous Edge Weights

**Hypothesis:** The previous experiments used binary edges (connected or not), discarding the continuous CHiCAGO interaction scores. Perhaps the topology *is* useful, but we need to model it as a continuous signal rather than binary.

**Design choices:**
- **All edges kept** (no threshold cutoff, min_score = 0) — the graph is now much denser
- **Edge weights:** raw CHiCAGO scores are log2-transformed (log2(score + 1)) then standardized to mean=0, std=1
- **Self-loops** receive neutral weight (0 after standardization)

**Two architectures tested:**

#### 3a. Weighted GCN (custom implementation)
Each neighbor's message is multiplied by the edge weight:

$$h_i' = \sum_{j \in \mathcal{N}(i)} \frac{w_{ij}}{\sqrt{\hat{d}_i \hat{d}_j}} \cdot W h_j$$

Simple and interpretable: strong CHiCAGO interactions contribute more to the aggregated message.

#### 3b. TransformerConv (PyTorch Geometric)
Edge features enter both the attention computation and value messages:

$$\alpha_{ij} = \text{softmax}\left(\frac{(W_q h_i)^T (W_k h_j + W_e e_{ij})}{\sqrt{d}}\right), \quad h_i' = \sum_j \alpha_{ij} (W_v h_j + W_e e_{ij})$$

The model learns *which* interaction strengths are informative — it doesn't assume a linear relationship between CHiCAGO score and importance.

**Why TransformerConv over standard GAT?** GAT computes attention purely from node features and ignores `edge_attr`. TransformerConv is one of the few PyG layers that natively incorporates edge features into both attention and message computation.

In [ ]:
print("=" * 70)
print("EXPERIMENT 3: Continuous Edge Weights — Comparison")
print("=" * 70)
display(Image(filename=f'{fig_base}/exp3_continuous_edges/comparison_chart.png', width=700))

In [ ]:
print("TransformerConv — Training Curves")
display(Image(filename=f'{fig_base}/exp3_continuous_edges/training_curves.png', width=900))

print("\nWeighted GCN — Training Curves (notice overfitting)")
display(Image(filename=f'{fig_base}/exp3_continuous_edges/training_curves_wgcn.png', width=900))

In [ ]:
print("TransformerConv — ROC and PR curves")
display(Image(filename=f'{fig_base}/exp3_continuous_edges/roc_pr_curves.png', width=800))

**Results:**

| Architecture | Params | AUROC | AUPRC | F1 | Precision | Recall |
|-------------|--------|-------|-------|-----|-----------|--------|
| Baseline GAT (binary edges) | 87,714 | 0.751 | 0.382 | 0.423 | 0.279 | 0.876 |
| Weighted GCN | 11,106 | 0.587 | 0.222 | 0.361 | 0.221 | 0.976 |
| **TransformerConv** | **336,930** | **0.766** | **0.399** | **0.445** | **0.306** | **0.816** |

**Interpretation:**

- **Weighted GCN failed** (AUROC = 0.587). With only 11K parameters, it was too simple to learn meaningful weight-feature interactions. The training curves show classic overfitting: val loss increases while train loss decreases. The model collapsed to predicting nearly everything as "activated" (recall = 0.98).

- **TransformerConv improved** all metrics over baseline. AUROC +1.5%, AUPRC +1.7%, and notably precision improved from 0.279 → 0.306 (+10% relative) with only a modest recall decrease. The continuous edge features allow the model to learn that strong CHiCAGO interactions carry different information than weak ones.

- The train-val gap in TransformerConv's training curves is smaller than the baseline, indicating less overfitting despite having 4× more parameters.

**Conclusion:** Architecture matters — the model must be expressive enough to learn from edge features. TransformerConv with continuous weights is our best architecture so far, but the improvement is modest (+1.7% AUPRC), suggesting topology still isn't the main bottleneck.

---

### Experiment 4: Feature Engineering (Best Model)

**Hypothesis:** With 85% isolated nodes, the GNN is essentially an MLP for most of the graph. Instead of trying to squeeze more signal from the sparse topology, we should enrich the node features themselves.

**Key insight:** The 3 raw ChIP signals don't explicitly encode the **epigenetic balance** that governs activation. A region with H3K27ac=5 and H3K27me3=5 (bivalent) is biologically very different from one with H3K27ac=5 and H3K27me3=0 (active) — but the raw features don't capture this ratio explicitly.

**Engineered features (10 total, from the same raw data):**

| # | Feature | Formula | Biological rationale |
|---|---------|---------|---------------------|
| 1 | H3K27ac | raw | Active enhancer/promoter mark |
| 2 | H3K4me2 | raw | Trithorax group mark |
| 3 | H3K27me3 | raw | Polycomb repressive mark |
| 4 | Active/Repressive ratio | log2(H3K27ac / H3K27me3) | **The key epigenetic switch** — high ratio = likely to activate |
| 5 | Trithorax/Polycomb ratio | log2(H3K4me2 / H3K27me3) | Bivalent domain resolution signal |
| 6 | Active enhancer ratio | log2(H3K27ac / H3K4me2) | Distinguishes active from poised |
| 7 | is_promoter | binary | Promoters and enhancers activate differently |
| 8 | Interaction degree | log2(degree + 1) | Connectivity in the interaction network |
| 9 | Mean neighbor H3K27ac | aggregated | Is this element in an active chromatin neighborhood? |
| 10 | Mean neighbor H3K27me3 | aggregated | Is this element in a Polycomb-repressed neighborhood? |

All features are standardized to mean=0, std=1. Ratio features use log2((|a| + ε) / (|b| + ε)) with ε=0.01 to handle zeros.

**Architecture:** TransformerConv with continuous edge weights (best from Experiment 3), now with 10 input features instead of 3.

In [ ]:
print("=" * 70)
print("EXPERIMENT 4: Enriched Features (Best Model) — Training Curves")
print("=" * 70)
display(Image(filename=f'{fig_base}/exp4_enriched/training_curves.png', width=900))

In [ ]:
print("ROC and Precision-Recall Curves")
display(Image(filename=f'{fig_base}/exp4_enriched/roc_pr_curves.png', width=800))

In [ ]:
print("Confusion Matrix")
display(Image(filename=f'{fig_base}/exp4_enriched/confusion_matrix.png', width=500))

In [ ]:
print("UMAP of Learned Node Embeddings")
display(Image(filename=f'{fig_base}/exp4_enriched/embeddings_umap.png', width=900))

In [ ]:
print("UMAP Colored by Histone Mark Levels")
display(Image(filename=f'{fig_base}/exp4_enriched/embeddings_umap_chipseq.png', width=900))

In [ ]:
print("Expression and Model Prediction Probability")
display(Image(filename=f'{fig_base}/exp4_enriched/embeddings_umap_expr_pred.png', width=700))

**Results:**

| Metric | Baseline (3 feat) | TransformerConv (3 feat) | **Enriched (10 feat)** | Improvement |
|--------|------------------|------------------------|---------------------|-------------|
| AUROC | 0.751 | 0.766 | **0.789** | +5.1% |
| AUPRC | 0.382 | 0.399 | **0.424** | +11.0% |
| F1 | 0.423 | 0.445 | **0.478** | +13.0% |
| Precision | 0.279 | 0.306 | **0.345** | +23.7% |
| Recall | 0.876 | 0.816 | 0.780 | -11.0% |
| Accuracy | 0.545 | 0.613 | **0.676** | +24.0% |
| Best epoch | 127 | 55 | **37** | Faster convergence |

**This is the best model.** Every metric improved substantially over the baseline. The most striking improvement is in **precision** (+23.7%), meaning the model produces significantly fewer false positives. The recall decrease is a healthy trade-off — the model is more selective in its predictions rather than flagging everything as "activated."

The model also converges much faster (epoch 37 vs 127), indicating the enriched features provide a clearer learning signal.

---

## 5. Complete Results Summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

experiments = pd.DataFrame({
    'Experiment': [
        '1. Baseline GAT\n(binary edges, th=5)',
        '2a. Lower threshold\n(th=3)',
        '2b. Connected only\n(th=5, connected)',
        '3a. Weighted GCN\n(continuous edges)',
        '3b. TransformerConv\n(continuous edges)',
        '4. Enriched features\n(10 feat, TransfConv)',
    ],
    'Features': [3, 3, 3, 3, 3, 10],
    'Edge_type': ['binary', 'binary', 'binary', 'continuous', 'continuous', 'continuous'],
    'AUROC': [0.751, 0.751, 0.744, 0.587, 0.766, 0.789],
    'AUPRC': [0.382, 0.383, 0.096, 0.222, 0.399, 0.424],
    'F1': [0.423, 0.421, 0.096, 0.361, 0.445, 0.478],
    'Precision': [0.279, 0.276, 0.051, 0.221, 0.306, 0.345],
    'Recall': [0.876, 0.885, 0.919, 0.976, 0.816, 0.780],
})

print(experiments[['Experiment', 'Features', 'AUROC', 'AUPRC', 'F1', 'Precision', 'Recall']].to_string(index=False))

In [ ]:
# Progression visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = range(len(experiments))
short_labels = ['1. Baseline\nGAT', '2a. th=3', '2b. Connected\nonly', 
                '3a. Weighted\nGCN', '3b. Transf.\nConv', '4. Enriched\nfeatures']

# --- Panel 1: AUROC + AUPRC ---
axes[0].plot(x, experiments['AUROC'], 'o-', color='#457B9D', linewidth=2.5, markersize=9, label='AUROC', zorder=5)
axes[0].plot(x, experiments['AUPRC'], 's-', color='#E63946', linewidth=2.5, markersize=9, label='AUPRC', zorder=5)
axes[0].axhline(y=0.190, color='gray', linestyle='--', alpha=0.5, label='Random AUPRC baseline')
axes[0].fill_between(x, experiments['AUPRC'], 0.190, alpha=0.1, color='#E63946')
axes[0].set_xticks(x)
axes[0].set_xticklabels(short_labels, fontsize=8)
axes[0].set_ylabel('Score', fontsize=11)
axes[0].set_title('Discrimination Metrics', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9, loc='lower left')
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3)

# --- Panel 2: Precision / Recall / F1 ---
axes[1].plot(x, experiments['Precision'], 'o-', color='#F4A261', linewidth=2.5, markersize=9, label='Precision')
axes[1].plot(x, experiments['Recall'], 's-', color='#2A9D8F', linewidth=2.5, markersize=9, label='Recall')
axes[1].plot(x, experiments['F1'], 'D-', color='#264653', linewidth=2.5, markersize=9, label='F1')
axes[1].set_xticks(x)
axes[1].set_xticklabels(short_labels, fontsize=8)
axes[1].set_ylabel('Score', fontsize=11)
axes[1].set_title('Classification Metrics', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)

# --- Panel 3: AUPRC bar chart with color coding ---
colors = ['#457B9D', '#ADB5BD', '#E76F51', '#E76F51', '#2A9D8F', '#06D6A0']
bars = axes[2].bar(x, experiments['AUPRC'], color=colors, edgecolor='white', linewidth=1.5)
axes[2].axhline(y=0.190, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
axes[2].set_xticks(x)
axes[2].set_xticklabels(short_labels, fontsize=8)
axes[2].set_ylabel('AUPRC', fontsize=11)
axes[2].set_title('AUPRC by Experiment', fontsize=13, fontweight='bold')
axes[2].set_ylim(0, 0.5)
axes[2].grid(axis='y', alpha=0.3)
axes[2].legend(fontsize=9)

# Add value labels on bars
for bar, val in zip(bars, experiments['AUPRC']):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.suptitle('Experimental Progression — GNN for Chromatin Activation Prediction', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/experiment_progression.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Ablation: What Contributed Most?

Decomposing the improvement from baseline (AUPRC = 0.382) to best model (AUPRC = 0.424):

| Change | AUPRC | Delta | % of total gain |
|--------|-------|-------|----------------|
| Baseline GAT, binary edges, 3 features | 0.382 | — | — |
| + Continuous edge weights | 0.382 → 0.399 | +0.017 | 40% |
| + TransformerConv architecture | (included above) | — | — |
| + Feature engineering (ratios, type, graph) | 0.399 → 0.424 | +0.025 | **60%** |
| **Total improvement** | | **+0.042** | **100%** |

**Feature engineering contributed ~60% of the total improvement**, despite adding no new raw data — only transformations of existing signals. This underscores that domain-specific feature design often outweighs architectural complexity in bioinformatics applications.

## 7. Biological Conclusions

### 7.1 Chromatin 3D topology at E8.5 is not predictive of activation

Across all experiments manipulating graph structure — binary vs continuous edges, different thresholds, connected-only subgraphs — topology contributed minimally to prediction. The GNN effectively functions as an MLP for 85% of nodes.

This implies that the enhancer-promoter contacts responsible for driving gene activation during CNCC migration **form dynamically during the transition itself**, rather than being pre-established at E8.5. This is consistent with models where Polycomb-mediated interactions are reorganized upon migration signals, and new activation-associated loops form de novo.

### 7.2 The H3K27ac/H3K27me3 balance is the primary predictor

The ratio features — particularly H3K27ac/H3K27me3 — provided the largest single improvement. This directly reflects the molecular mechanism of activation: Polycomb eviction (loss of H3K27me3) and acetyltransferase recruitment (gain of H3K27ac). Elements where this balance already favors activation at E8.5 are most likely to complete the switch by E10.5.

### 7.3 Local chromatin neighborhood provides modest additional signal

Mean neighbor H3K27ac and H3K27me3 levels contributed to the enriched model, suggesting some spatial coordination: elements embedded in active chromatin neighborhoods are slightly more likely to activate. This is consistent with the concept of chromatin domains and spreading of active marks.

### 7.4 Promoters and enhancers follow distinct activation programs

The `is_promoter` binary feature improved the model, confirming that these regulatory element classes activate through different mechanisms despite sharing the same histone marks. This aligns with known biology: promoters are often pre-loaded with Pol II (paused polymerase), while enhancers require de novo transcription factor binding.

## 8. Limitations & Future Directions

### 8.1 Limitations

**Data limitations:**
- Only 3 histone marks available. Additional epigenomic data (H3K4me1, H3K4me3, ATAC-seq, DNA methylation, motif scores) would likely improve performance substantially, as each captures different aspects of the regulatory landscape.
- Expression data excluded because it is only valid for promoters. A promoter-specific model incorporating expression could perform better on that subset.
- CHiCAGO interactions are bulk measurements from a heterogeneous cell population. Single-cell Hi-C would provide cell-type-specific interaction maps.

**Methodological limitations:**
- The graph is extremely sparse at any reasonable CHiCAGO threshold. PCHi-C inherently captures only promoter-baited interactions — enhancer-enhancer interactions, which might carry additional activation signal, are missing.
- Self-loops allow isolated nodes to be processed, but they still receive no neighborhood information. Alternative approaches like virtual edges (connecting genomically proximal elements) could help.
- Class imbalance (1:4.3) limits achievable precision. More sophisticated sampling strategies (SMOTE for graphs, focal loss) could help.
- The model predicts from a single snapshot (E8.5). Time-series approaches capturing the E8.5 → E10.5 dynamic would be more powerful.

**Interpretation limitations:**
- The conclusion that "topology doesn't help" is specific to *pre-migration* topology. The contacts that drive activation may form *during* migration and would only be visible in E10.5 data.
- We tested topology through GNN message passing. Other methods of encoding topology (e.g., graph-level features like betweenness centrality, community membership) might extract different signals.

### 8.2 Future Directions

1. **Additional epigenomic features**: H3K4me1 (enhancer mark), ATAC-seq (accessibility), CpG density, transcription factor motif scores
2. **Temporal modeling**: Include both E8.5 and E10.5 features and predict the *change* rather than the endpoint
3. **Virtual edges**: Connect genomically proximal elements (within same TAD or within linear distance) to create a denser, biologically motivated graph
4. **Heterogeneous GNN**: Use different message-passing rules for promoter→enhancer, enhancer→promoter, and promoter→promoter interactions
5. **Explainability**: Apply GNNExplainer or attention weight analysis to identify which interactions and features are most important for individual predictions
6. **Transfer learning**: Test whether the model trained on E8.5→E10.5 generalizes to other developmental transitions (e.g., E10.5→E14.5)

## 9. Technical Stack

| Component | Library |
|-----------|--------|
| Deep learning | PyTorch |
| Graph neural networks | PyTorch Geometric |
| GNN layers | GATConv, TransformerConv, custom WeightedGCNConv |
| Evaluation | scikit-learn (AUROC, AUPRC, F1, confusion matrix) |
| Dimensionality reduction | UMAP |
| Visualization | matplotlib, seaborn |
| Configuration | PyYAML |
| Data | SummarizedExperiment (R/Bioconductor) → TSV → PyTorch Geometric Data |

---

## Appendix: Key Code Components

In [ ]:
# Feature engineering — the most impactful component

import numpy as np

def safe_ratio(a, b, eps=0.01):
    """Log2 ratio with epsilon to handle zeros."""
    return np.log2((np.abs(a) + eps) / (np.abs(b) + eps))

def engineer_features(nodes_df, feat_cols, src_indices, dst_indices):
    """
    Transform 3 raw ChIP-seq signals into 10 biologically motivated features.
    
    Parameters
    ----------
    nodes_df : DataFrame with ChIP-seq columns and node_type
    feat_cols : list of 3 column names [H3K27ac, H3K4me2, H3K27me3]
    src_indices, dst_indices : edge source/destination arrays
    
    Returns
    -------
    X : np.ndarray of shape (n_nodes, 10), standardized
    """
    n = len(nodes_df)
    h3k27ac = nodes_df[feat_cols[0]].values.astype(np.float32)
    h3k4me2 = nodes_df[feat_cols[1]].values.astype(np.float32)
    h3k27me3 = nodes_df[feat_cols[2]].values.astype(np.float32)
    
    # 1. Raw ChIP signals (3 features)
    raw = np.column_stack([h3k27ac, h3k4me2, h3k27me3])
    
    # 2. Ratio features (3 features)
    ratios = np.column_stack([
        safe_ratio(h3k27ac, h3k27me3),   # active vs repressive balance
        safe_ratio(h3k4me2, h3k27me3),   # trithorax vs polycomb
        safe_ratio(h3k27ac, h3k4me2),    # active enhancer signal
    ])
    
    # 3. Node type (1 feature)
    is_prom = (nodes_df['node_type'] == 'promoter').values.astype(np.float32).reshape(-1, 1)
    
    # 4. Graph topology features (3 features)
    degrees = np.zeros(n, dtype=np.float32)
    nbr_ac = np.zeros(n, dtype=np.float32)
    nbr_me3 = np.zeros(n, dtype=np.float32)
    nbr_count = np.zeros(n, dtype=np.float32)
    
    for s, d in zip(src_indices, dst_indices):
        degrees[s] += 1
        nbr_ac[s] += h3k27ac[d]
        nbr_me3[s] += h3k27me3[d]
        nbr_count[s] += 1
    
    nbr_count[nbr_count == 0] = 1
    graph_feats = np.column_stack([
        np.log2(degrees + 1),
        nbr_ac / nbr_count,
        nbr_me3 / nbr_count,
    ])
    
    # Combine and standardize
    X = np.hstack([raw, ratios, is_prom, graph_feats])
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
    
    feature_names = [
        'H3K27ac', 'H3K4me2', 'H3K27me3',
        'ratio_ac/me3', 'ratio_me2/me3', 'ratio_ac/me2',
        'is_promoter',
        'log2_degree', 'mean_nbr_H3K27ac', 'mean_nbr_H3K27me3',
    ]
    return X, feature_names

print(f"Feature engineering: 3 raw signals → 10 features")
print(f"Groups: ChIP (3) + Ratios (3) + Node type (1) + Graph topology (3)")

In [ ]:
# TransformerConv model — best architecture

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv, BatchNorm

class ChromatinTransformer(nn.Module):
    """
    Graph Transformer for chromatin activation prediction.
    
    Key design choices:
    - TransformerConv: natively uses edge features (CHiCAGO scores)
    - 3 layers: captures 3-hop neighborhood information
    - 4 attention heads: learns different attention patterns in parallel
    - BatchNorm + ELU: stable training with non-saturating activation
    - Dropout 0.3: regularization for the sparse graph
    - 2-layer classifier head: separates representation from classification
    """
    def __init__(self, in_channels, hidden=64, layers=3, heads=4, 
                 dropout=0.3, edge_dim=1):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(TransformerConv(
            in_channels, hidden, heads=heads, edge_dim=edge_dim, dropout=dropout))
        self.bns.append(BatchNorm(hidden * heads))
        
        for _ in range(layers - 2):
            self.convs.append(TransformerConv(
                hidden * heads, hidden, heads=heads, edge_dim=edge_dim, dropout=dropout))
            self.bns.append(BatchNorm(hidden * heads))
        
        self.convs.append(TransformerConv(
            hidden * heads, hidden, heads=1, concat=False, 
            edge_dim=edge_dim, dropout=dropout))
        self.bns.append(BatchNorm(hidden))
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden // 2, 2)
        )
        self.dropout = dropout
    
    def forward(self, x, edge_index, edge_attr=None):
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
            if i < len(self.convs) - 1:
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)
    
    def get_embeddings(self, x, edge_index, edge_attr=None):
        """Return node embeddings before classifier (for UMAP)."""
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.elu(x)
        return x

model = ChromatinTransformer(in_channels=10)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Architecture: 3 TransformerConv layers → 2-layer MLP classifier")

In [ ]:
# Class-weighted training — critical for imbalanced data

def compute_class_weights(y, mask):
    """Inverse-frequency weights."""
    labels = y[mask]
    counts = torch.bincount(labels)
    weights = 1.0 / counts.float()
    return weights / weights.sum() * len(weights)

print("Training strategy:")
print("  Loss:       Class-weighted cross-entropy")
print("  Optimizer:  Adam (lr=0.001, weight_decay=5e-4)")
print("  Scheduler:  ReduceLROnPlateau (patience=10, factor=0.5)")
print("  Stopping:   Early stopping on val AUPRC (patience=30)")
print("  Learning:   Transductive — full graph in every forward pass")

---

*This analysis demonstrates systematic experimental methodology applied to a real biological question: formulating hypotheses, testing them rigorously, and drawing conclusions from both positive and negative results.*